# R01. Original vs revised DE audit

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd.parents[1] if cwd.name == "revision" else cwd
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import *
from reproducibility import seed_everything

seed_everything()

print("PROJECT_ROOT =", PROJECT_ROOT)
print("GLOBAL_SEED =", GLOBAL_SEED)


PROJECT_ROOT = /Users/jihopark/Desktop/MCDA_revision_final
GLOBAL_SEED = 20260829


In [2]:
import subprocess, sys

res = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "src" / "R01_original_vs_revised_DE.py")],
    text=True,
    capture_output=True
)

print(res.stdout)

if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Revision audit failed.")


Original set  Found  Still exploratory  BH-FDR significant
      8 DEMs      8                  8                   1
     23 DEGs     23                  5                   0

Saved revision audit to: /Users/jihopark/Desktop/MCDA_revision_final/results/revision



In [3]:
import pandas as pd

display(pd.read_csv(REVISION_RESULTS_DIR / "R01_revision_summary.csv"))


,Original set,Found,Still exploratory,BH-FDR significant
0,8 DEMs,8,8,1
1,23 DEGs,23,5,0


In [5]:
import pandas as pd

deg_cmp = pd.read_csv(
    REVISION_RESULTS_DIR
    / "R01_original_23_DEG_vs_revised_limma.csv"
)

dem_cmp = pd.read_csv(
    REVISION_RESULTS_DIR
    / "R01_original_8_DEM_vs_revised_limma.csv"
)

print("Original DEG comparison:", deg_cmp.shape)
print("Original DEM comparison:", dem_cmp.shape)

retained_old_deg = deg_cmp.loc[
    deg_cmp["exploratory_candidate"],
    [
        "Gene",
        "logFC",
        "P.Value",
        "adj.P.Val",
        "diff_MZ1",
        "diff_MZ2",
        "diff_MZ3",
        "Revised status",
    ]
].copy()

display(retained_old_deg)

significant_old_dem = dem_cmp.loc[
    dem_cmp["FDR_significant"],
    [
        "miRNA",
        "logFC",
        "P.Value",
        "adj.P.Val",
        "diff_MZ1",
        "diff_MZ2",
        "diff_MZ3",
        "Revised status",
    ]
].copy()

display(significant_old_dem)

retained_old_deg.to_csv(
    REVISION_RESULTS_DIR / "R01_retained_original_DEGs.csv",
    index=False
)

significant_old_dem.to_csv(
    REVISION_RESULTS_DIR / "R01_FDR_significant_original_DEMs.csv",
    index=False
)

print("Saved to:", REVISION_RESULTS_DIR)


Original DEG comparison: (23, 14)
Original DEM comparison: (8, 14)


,Gene,logFC,P.Value,adj.P.Val,diff_MZ1,diff_MZ2,diff_MZ3,Revised status
0,TOMM40L,-0.703597,0.000697,0.965865,-0.65912,-0.58929,-0.86238,Exploratory only
1,PHC1,-0.682983,0.001078,0.965865,-0.49818,-0.65781,-0.89296,Exploratory only
2,CISD1,0.616583,0.001574,0.965865,0.65396,0.71455,0.48124,Exploratory only
3,SGOL1,-0.624573,0.002149,0.965865,-0.69923,-0.37481,-0.79968,Exploratory only
4,PRKX,-0.587287,0.002212,0.965865,-0.69543,-0.44191,-0.62452,Exploratory only


,miRNA,logFC,P.Value,adj.P.Val,diff_MZ1,diff_MZ2,diff_MZ3,Revised status
0,hsa-miR-1292-5p,-1.66582,4.060170e-07,0.001795,-1.62401,-1.58455,-1.7889,FDR-significant


Saved to: /Users/jihopark/Desktop/MCDA_revision_final/results/revision


## Next Step: miRNA–mRNA Integration

Based on the revised differential expression analysis, downstream miRNA–mRNA integration will distinguish between **confirmatory** and **exploratory** evidence.

### Confirmatory analysis
- **1 FDR-significant miRNA:** hsa-miR-1292-5p
- Candidate target genes will be searched among the **114 exploratory mRNA candidates**.

### Exploratory analysis
- **30 exploratory miRNA candidates**
- **114 exploratory mRNA candidates**
- Putative miRNA–mRNA regulatory relationships will be cross-referenced using:
  - miRDB
  - TargetScan
  - miRTarBase

Database-supported miRNA–mRNA pairs will be identified by intersecting predicted or validated target genes with the revised mRNA candidate set.

Expression direction will also be considered:
- **Anti-correlated:** miRNA and mRNA change in opposite directions
- **Same-direction:** miRNA and mRNA change in the same direction

All miRNA–mRNA relationships will be interpreted as **exploratory database-supported associations**, not as experimentally validated regulatory interactions.